# Product Engineering Lab · CadFlow B-rep reconstruction

This notebook builds the component-level OpenCascade B-rep reconstruction for the GB2591502A patent architecture using CadFlow. It exports separate STEP/STL/GLB files plus an all-components assembly compound. It is a research proxy, not original Nyfasi CAD or a verified retail digital twin.

CadFlow is pinned to the same source commit as the repository's GitHub Actions workflow. The runtime installation includes native build prerequisites and OpenCascade 7.9.3.1.


In [ ]:
import sys, platform
print(sys.version)
if sys.version_info < (3, 10) or sys.version_info >= (3, 14):
    raise RuntimeError('CadFlow supports Python 3.10 through 3.13.')
print('Platform:', platform.platform())


In [ ]:
import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'python3-dev'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'cadquery-ocp==7.9.3.1'], check=True)
CADFLOW_COMMIT = '43fbac0515a560229d700e07281cc5c08c3f9765'
cadflow_source = f'git+https://github.com/yhz5613813/CadFlow.git@{CADFLOW_COMMIT}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', cadflow_source], check=True)
import cadflow as cad
print('CadFlow:', cad.__version__)


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/product-engineering-lab')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/andiekobbietks/product-engineering-lab.git', str(repo)], check=True)
experiment = repo / 'experiments/001-nyfasi-reconstruction'
scripts = experiment / 'scripts'
subprocess.run([sys.executable, str(scripts / 'validate_provenance.py')], cwd=repo, check=True)
subprocess.run([sys.executable, str(scripts / 'validate_geometry.py')], cwd=repo, check=True)
print('Repository and parameter checks passed.')


In [ ]:
build_script = scripts / 'build_cadflow_reconstruction.py'
subprocess.run([sys.executable, str(build_script)], cwd=repo, check=True)
output = experiment / 'output/cadflow'
print('Generated files:', len(list(output.iterdir())))
print('Assembly STEP:', (output / 'nyfasi_research_assembly.step').stat().st_size, 'bytes')
print('Assembly GLB:', (output / 'nyfasi_research_assembly.glb').stat().st_size, 'bytes')


In [ ]:
import json
manifest = json.loads((output / 'cadflow-build-manifest.json').read_text())
print('Component count:', manifest['component_count'])
for component_id, record in manifest['components'].items():
    print(component_id, 'volume_mm3=', round(record['volume_mm3'], 3), 'bbox_mm=', [round(v, 3) for v in record['bbox_mm']])
print('Assembly validation:', manifest['assembly']['compound']['validation'])
print('Limitations:')
for item in manifest['limitations']:
    print('-', item)


In [ ]:
import shutil
from google.colab import files
archive_base = Path('/content/nyfasi-cadflow-brep')
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=output)
files.download(archive)


## Engineering boundary

CadFlow validation establishes that each generated B-rep is structurally valid according to the geometry kernel. It does not establish dimensional accuracy, correct patent interpretation, fit, clearances, seal compression, material behaviour, durability or manufacturing suitability.

The Sand3D Hugging Face datasets are separate reference collections for procedural-program patterns. They do not contain verified Nyfasi CAD. Consult the repository's [CadFlow B-rep documentation](https://github.com/andiekobbietks/product-engineering-lab/blob/main/docs/experiments/cadflow-brep.md) and [Sand3D dataset strategy](https://github.com/andiekobbietks/product-engineering-lab/blob/main/docs/experiments/sand3d-dataset-strategy.md).
